<a href="https://colab.research.google.com/github/rafidfajar/data-science-2026/blob/main/Pertemuan12_Muhamad_Rafid_Fajar_250401020195.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning"
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
 n_item = np.random.randint(2, 6)
 transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
 if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
    transaksi[i].append('Selai')
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))


Contoh transaksi: [[np.str_('Selai'), np.str_('Susu'), np.str_('Mentega')], [np.str_('Gula'), np.str_('Roti'), np.str_('Teh'), np.str_('Selai'), np.str_('Sereal')], [np.str_('Mentega'), np.str_('Susu'), np.str_('Selai')]]
Jumlah transaksi: 50


In [17]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

In [18]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
 freq = apriori(df, min_support=ms, use_colnames=True)
 print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))


min_support=0.05: 85 itemset ditemukan
min_support=0.1: 46 itemset ditemukan
min_support=0.2: 15 itemset ditemukan
   support   itemsets
5     0.52    (Selai)
1     0.44     (Keju)
3     0.42  (Mentega)
6     0.40   (Sereal)
7     0.40     (Susu)
0     0.36     (Gula)
9     0.32    (Telur)
2     0.28     (Kopi)
4     0.28     (Roti)
8     0.28      (Teh)


In [22]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
 min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
 'support', 'confidence', 'lift']].head(10))
# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?


         antecedents consequents  support  confidence      lift
12     (Selai, Susu)   (Mentega)     0.14    0.636364  1.515152
15     (Gula, Selai)   (Mentega)     0.10    0.625000  1.488095
11  (Mentega, Selai)      (Susu)     0.14    0.583333  1.458333
1             (Susu)   (Mentega)     0.24    0.600000  1.428571
0          (Mentega)      (Susu)     0.24    0.571429  1.428571
4            (Telur)      (Keju)     0.20    0.625000  1.420455
9             (Kopi)    (Sereal)     0.14    0.500000  1.250000
16   (Selai, Sereal)      (Susu)     0.10    0.500000  1.250000
17    (Sereal, Susu)     (Selai)     0.10    0.625000  1.201923
14   (Gula, Mentega)     (Selai)     0.10    0.625000  1.201923


Aturan mana yang paling kuat (Lift tertinggi)?
(Selai, Susu) dengan consequents(Mentega), dengan lift 1,52 dan confidence 0,64.

Apakah masuk akal secara bisnis?
Ya, cukup masuk akal Selai, Susu, dan Mentega sama-sama produk sarapan/dairy yang wajar dibeli bersamaan

In [20]:
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
 idx = katalog.index[katalog['produk'] == nama_produk][0]
 skor = list(enumerate(sim_matrix[idx]))
 skor = sorted(skor, key=lambda x: x[1], reverse=True)
 skor = [s for s in skor if s[0] != idx][:top_n]
 return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))


Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [23]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
 lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?


Rekomendasi dari Association Rules:
  consequents      lift
7     (Selai)  1.098901
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


**Apakah kedua pendekatan konsisten?**
Tidak sepenuhnya. Association rules merekomendasikan produk berdasarkan **pola pembelian aktual** (mis. Roti → Selai berdasarkan transaksi nyata), sedangkan content-based merekomendasikan berdasarkan **kemiripan kategori** (Roti → Selai, Sereal, Susu, karena sama-sama Bakery/Dairy). Selai kebetulan muncul di kedua metode, tapi Sereal dan Susu hanya muncul di content-based — jadi hasilnya bisa berbeda arah tergantung metode yang dipakai.

**Kapan pakai yang mana / hybrid?**
- **Association rules**: cocok kalau data transaksi banyak dan pola beli-bersama sudah teruji secara statistik.
- **Content-based**: cocok untuk produk baru atau *cold-start* yang belum punya riwayat transaksi.
- **Hybrid**: gabungkan keduanya — utamakan association rules jika datanya kuat, fallback ke content-based jika data transaksi tipis atau produk baru. Ini pendekatan umum di sistem rekomendasi nyata.